# Phase 3 Real-Data Cleaning

Purpose: transition the statistical baseline from synthetic/pre-data validation to empirical real participant data preparation.

Scope: this notebook preserves the immutable raw workbook, audits the raw schema, reconstructs trial-level randomisation from the submitted JSON payloads, attaches the frozen Phase 3 acoustic features, derives trial-level preferred mixes without breaking ties, and writes canonical real-data cleaning outputs.

No inferential modelling, Bambi/PyMC fitting, statistical comparison, winsorisation, normalisation, or subjective participant exclusion is performed here.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif not (PROJECT_ROOT / "statistical-baseline").exists():
    PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "statistical-baseline").exists())

sys.path.insert(0, str(PROJECT_ROOT / "statistical-baseline" / "src"))
from statistical_baseline.real_data_cleaning import OUTPUT_DIR, RAW_SOURCE, run_cleaning

PROJECT_ROOT, RAW_SOURCE, OUTPUT_DIR

(WindowsPath('C:/Users/oscar/Documents/7. QMUL UNIVERSITY/1. Master Program/3. MSc Project/intent2control-dissertation'),
 WindowsPath('C:/Users/oscar/Documents/7. QMUL UNIVERSITY/1. Master Program/3. MSc Project/intent2control-dissertation/statistical-baseline/data/real/raw/listening_preference_responses_33_immutable.xlsx'),
 WindowsPath('C:/Users/oscar/Documents/7. QMUL UNIVERSITY/1. Master Program/3. MSc Project/intent2control-dissertation/statistical-baseline/data/real'))

## Execute Cleaning Pipeline

The cleaning code reads only from the immutable raw workbook stored under `statistical-baseline/data/real/raw/`. All derived tables are regenerated from code.

In [2]:
outputs = run_cleaning()
summary = outputs["summary"]
validation = outputs["validation"]
manifest = outputs["manifest"]
summary

,raw_submissions,raw_columns,unit_of_observation_raw,group_01_submissions,group_02_submissions,complete_participants,incomplete_participants,possible_duplicates,probable_duplicates,manual_review_cases,...,episodes_found,songs_found,stimuli_found,all_randomisation_mappings_reconstructed,participant_song_episode_trials,trials_with_rating_ties,trial_tie_proportion,all_stimuli_mapped_to_features,z_features_from_frozen_phase3_table,final_gate
0,33,40,one row per participant/submission with trial-...,17,16,33,0,0,0,0,...,EDR-1|EDR-2|FM-1,id_like_to_know|in_the_meantime|lead_me|pourin...,20,True,198,16,0.080808,True,True,REAL DATA READY FOR STATISTICAL MODELLING


## Raw-Data Provenance and Final Design

In [3]:
manifest["raw_provenance"], manifest["raw_shape"], manifest["sheet_audit"], manifest["design"]

({'original_filename': 'listening_preference_responses_33.xlsx',
  'stored_path': 'statistical-baseline\\data\\real\\raw\\listening_preference_responses_33_immutable.xlsx',
  'copied_this_run': True,
  'sha256': '5bab388fbf564e0caf5c1ca8a5a722bf8d517e23c018e48375d076e75dba0bdd',
  'bytes': 294316},
 {'rows': 33, 'columns': 40},
 [{'sheet_name': 'listening-study-5mix', 'row_count': 33, 'column_count': 40},
  {'sheet_name': 'Hoja1', 'row_count': 0, 'column_count': 0}],
 {'stimulus_configuration_version': 'five_mix_frontend_v1_2026-08-06',
  'study_version_expected': 'five_mix_frontend_v1_2026-08-06',
  'schema_version_expected': 'five_mix_netlify_forms_v1',
  'group_count': 2,
  'songs_total': 4,
  'stimuli_total': 20,
  'episodes': ['EDR-1', 'EDR-2', 'FM-1'],
  'ratings_per_participant': 30,
  'comments_per_participant': 6,
  'comment_unit': "one comparative comment per participant x song x episode trial, repeated on each of that trial's five rating rows"})

## Raw Schema Audit

In [4]:
outputs["schema_audit"][["column_name", "dtype", "missing_count", "unique_non_missing", "column_role", "json_parse_errors"]]

,column_name,dtype,missing_count,unique_non_missing,column_role,json_parse_errors
0,study_id,object,0,33,participant_or_group_identifier,0
1,study_version,object,0,1,form_or_backend_metadata,0
2,schema_version,object,0,1,form_or_backend_metadata,0
3,stimulus_configuration_version,object,0,1,form_or_backend_metadata,0
4,source_version,object,0,1,form_or_backend_metadata,0
5,submission_status,object,0,1,form_or_backend_metadata,0
6,study_group,object,0,2,participant_or_group_identifier,0
7,group_id,object,0,2,participant_or_group_identifier,0
8,started_at,datetime64[ns],1,32,timestamp,0
9,completed_at,datetime64[ns],0,33,timestamp,0


## Submission and Participant Completeness Audit

In [5]:
outputs["exclusion_log"][[
    "participant_id", "group", "n_episodes_completed", "n_song_episode_trials_completed",
    "n_ratings", "n_comments", "missing_ratings", "missing_comments",
    "duplicated_trial_rating_combinations", "duplicate_classification",
    "complete_submission", "include_recommended", "requires_manual_review", "exclusion_reason"
]]

,participant_id,group,n_episodes_completed,n_song_episode_trials_completed,n_ratings,n_comments,missing_ratings,missing_comments,duplicated_trial_rating_combinations,duplicate_classification,complete_submission,include_recommended,requires_manual_review,exclusion_reason
0,P033,group_02,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
1,P032,group_02,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
2,P031,group_01,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
3,P030,group_02,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
4,P029,group_02,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
5,P028,group_01,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
6,P027,group_02,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
7,P026,group_02,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
8,P025,group_01,3,6,30,30,0,0,0,no duplicate concern,True,True,False,
9,P024,group_01,3,6,30,30,0,0,0,no duplicate concern,True,True,False,


## Rating, Comment, Randomisation, and Feature Validation

In [6]:
validation

,check,passed,details
0,raw_workbook_has_rows,True,
1,one_wide_row_per_submission,True,
2,all_groups_valid,True,
3,all_group_song_combinations_valid,True,
4,all_episodes_valid,True,
5,all_songs_expected,True,
6,all_stimuli_expected,True,
7,all_randomisation_mappings_reconstructed,True,
8,no_invalid_ratings,True,
9,no_missing_comments_on_rating_rows,True,


## Derived Trial Preferences and Ties

In [7]:
outputs["trial_preferences"].head(), outputs["ties"].head(), outputs["summary"][["participant_song_episode_trials", "trials_with_rating_ties", "trial_tie_proportion"]]

(  participant_id     group          song_id episode        trial_id  \
 0           P001  group_01  id_like_to_know   EDR-1  P001__trial_06   
 1           P001  group_01  id_like_to_know   EDR-2  P001__trial_03   
 2           P001  group_01  id_like_to_know    FM-1  P001__trial_02   
 3           P001  group_01          lead_me   EDR-1  P001__trial_05   
 4           P001  group_01          lead_me   EDR-2  P001__trial_04   
 
    trial_index  candidate_count  unique_candidate_count  rating_count  \
 0            6                5                       5             5   
 1            3                5                       5             5   
 2            2                5                       5             5   
 3            5                5                       5             5   
 4            4                5                       5             5   
 
    comment_count  distinct_comment_count  max_rating  n_tied_winners  \
 0              5                       1      

## Participant Metadata Audit

In [8]:
outputs["metadata_audit"]

,metadata_variable,raw_representation,count,missing_count,unique_values,proposed_analysis_representation
0,age_range,25_34,24,0,5,preserve as collected; trimmed whitespace only
1,age_range,18_24,6,0,5,preserve as collected; trimmed whitespace only
2,age_range,35_44,1,0,5,preserve as collected; trimmed whitespace only
3,age_range,45_54,1,0,5,preserve as collected; trimmed whitespace only
4,age_range,55_64,1,0,5,preserve as collected; trimmed whitespace only
5,gender,man,17,1,3,preserve as collected; trimmed whitespace only
6,gender,woman,15,1,3,preserve as collected; trimmed whitespace only
7,gender,<missing>,1,1,3,preserve as collected; trimmed whitespace only
8,cultural_influence_country,Mexico,4,0,27,preserve as collected; trimmed whitespace only
9,cultural_influence_country,Japan,2,0,27,preserve as collected; trimmed whitespace only


## Canonical Outputs and Final Gate

In [9]:
print(summary.loc[0, "final_gate"])
for label, path in manifest["outputs"].items():
    print(f"{label}: {PROJECT_ROOT / path}")

REAL DATA READY FOR STATISTICAL MODELLING
ratings: C:\Users\oscar\Documents\7. QMUL UNIVERSITY\1. Master Program\3. MSc Project\intent2control-dissertation\statistical-baseline\data\real\real_ratings_clean.csv
participants: C:\Users\oscar\Documents\7. QMUL UNIVERSITY\1. Master Program\3. MSc Project\intent2control-dissertation\statistical-baseline\data\real\real_participants_clean.csv
trial_preferences: C:\Users\oscar\Documents\7. QMUL UNIVERSITY\1. Master Program\3. MSc Project\intent2control-dissertation\statistical-baseline\data\real\real_trial_preferences.csv
ties: C:\Users\oscar\Documents\7. QMUL UNIVERSITY\1. Master Program\3. MSc Project\intent2control-dissertation\statistical-baseline\data\real\real_trial_ties_long.csv
submission_audit: C:\Users\oscar\Documents\7. QMUL UNIVERSITY\1. Master Program\3. MSc Project\intent2control-dissertation\statistical-baseline\data\real\real_submission_audit.csv
exclusion_log: C:\Users\oscar\Documents\7. QMUL UNIVERSITY\1. Master Program\3. MSc